# 03 — Discovery: sources, RSS and Tavily (free-tier safe)

**What this tests:** finding opportunities from real public sources — `sources.yaml`,
RSS feeds and Tavily search — while per-run quotas, robots.txt and failures are all
handled.

| step | offline (`LLM_PROVIDER=fake` / no keys) | with real keys |
|---|---|---|
| `sources.yaml` | read from disk | same |
| RSS | parses `data/samples/feed_opportunities.xml` | fetches live feeds (UA + delay + robots) |
| Tavily | loads `data/samples/tavily_response_sample.json`, labeled OFFLINE | live search, disk-cached |

Every source row carries `verified` + `notes`; every remote call is accounted
(`made` real / `cached` free / `skipped` over-limit), and a dead feed or missing
key becomes a message — never an exception.

In [ ]:
%load_ext autoreload
%autoreload 2

import json
import sys
from pathlib import Path

# Make oppradar_ai importable whether the kernel starts in ai/ or ai/notebooks/.
AI_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(AI_ROOT) not in sys.path:
    sys.path.insert(0, str(AI_ROOT))

import pandas as pd
from IPython.display import display

from oppradar_ai.config import get_settings, setup_logging
from oppradar_ai.ingestion import (
    RunBudget,
    SearchHit,
    TavilySearcher,
    load_sources,
    read_feed,
)

setup_logging('INFO')
settings = get_settings().ensure_dirs()
SAMPLES = settings.samples_dir

# REAL_MODE = real Gemini + Tavily keys configured -> live Tavily/RSS.
# Otherwise every step uses committed local fixtures (deterministic, offline).
REAL_MODE = settings.llm_provider == 'gemini' and bool(settings.tavily_key_value)
print('LLM provider:', settings.llm_provider, '| live Tavily/RSS:', 'yes' if REAL_MODE else 'no')

## Step 1 — sources.yaml: real sources only, with provenance

`verified: true` = the URL was confirmed reachable (date inside `notes`);
`verified: false` = could not be confirmed and is kept **disabled** until you check it
manually. The loader refuses rows without `notes`/`verified` or with non-http URLs.

In [ ]:
sources = load_sources()
sources_df = pd.DataFrame([
    {
        'source': s.name,
        'kind': s.kind,
        'url': s.url,
        'enabled': s.enabled,
        'verified': s.verified,
        'notes': s.notes[:100] + ('...' if len(s.notes) > 100 else ''),
    }
    for s in sources
])
display(sources_df)
print(f'{len(sources)} sources | verified {sum(s.verified for s in sources)} '
      f'| enabled {sum(s.enabled for s in sources)}')
print('need manual check:', [s.name for s in sources if not s.verified] or 'none')

## Step 2 — RSS reading (fixtures offline, live feeds online)

Live mode goes through `PageFetcher`: clear `User-Agent`, a polite delay between
requests, robots.txt enforcement (unreachable robots = allow; disallow = skipped and
reported). Offline mode parses the committed sample feed — both paths return
`(entries, errors)`, never an exception.

In [ ]:
feed_source = next((s for s in sources if s.kind == 'rss' and s.enabled), None)
entries, errors = [], []

if REAL_MODE and feed_source is not None:
    entries, errors = read_feed(feed_source.url)
    print(f'live {feed_source.name}: {len(entries)} entries, {len(errors)} errors')
    if errors:
        print('  -> falling back to the sample feed (no crash)')

if not entries:
    entries, errors = read_feed(str(SAMPLES / 'feed_opportunities.xml'))
    print(f'sample feed: {len(entries)} entries, {len(errors)} errors')

feed_df = pd.DataFrame([
    {
        'title': e.title[:58],
        'published': e.published,
        'deadline phrase in summary': 'yes' if 'deadline' in e.summary.lower() else 'no',
        'url': e.url,
    }
    for e in entries
])
display(feed_df)
if errors:
    print('errors:', errors)

## Step 3 — Tavily search: cache first, quota second, network last

1. a disk-cache hit is **free** and never consumes the quota;
2. a real call must fit inside `TAVILY_MAX_SEARCHES_PER_RUN` (default 5) — over the cap
   the query is `skipped`, not sent;
3. a missing key or a failing API is recorded as a failure and returns `[]`.

Offline mode never touches the network: it reads the committed sample response and labels it.

In [ ]:
budget = RunBudget(settings)
searcher = TavilySearcher(settings, budget=budget)
query = 'scholarships for students in Algeria 2027'

if REAL_MODE:
    hits = searcher.search(query)
else:
    sample = json.loads((SAMPLES / 'tavily_response_sample.json').read_text(encoding='utf-8'))
    hits = [
        SearchHit(title=r['title'], url=r['url'], snippet=r['content'], score=r.get('score', 0.0))
        for r in sample['results']
    ]
    print('OFFLINE: loaded data/samples/tavily_response_sample.json (no network)')

hits_df = pd.DataFrame(
    [{'title': h.title[:60], 'score': round(h.score, 3), 'url': h.url} for h in hits]
)
display(hits_df)
print('usage:', searcher.usage.model_dump())

## Step 4 — Free-tier accounting: used vs cached vs skipped

The same query again is a `cached += 1` with **zero** network. Then a deliberately tiny
budget (`max_searches=1`) proves the cap: the second query is `skipped`, never sent.
Offline the "API" is a stub over the sample file, so the demo costs nothing either way.

In [ ]:
if REAL_MODE:
    searcher.search(query)  # same query -> disk-cache hit (free)
    print('after repeat:', searcher.usage.model_dump())

if REAL_MODE:
    stub_api = None
else:
    _sample = json.loads((SAMPLES / 'tavily_response_sample.json').read_text(encoding='utf-8'))

    def stub_api(q, n):  # offline stand-in for the Tavily API
        return _sample

tiny = RunBudget(settings, max_searches=1)
capped = TavilySearcher(settings, budget=tiny, api_call=stub_api)
capped.search('demo query one for the cap')
capped.search('demo query two for the cap')
print('tiny budget (limit 1):', tiny.searches.model_dump())

## Expected output

- **sources table**: 6 rows — `verified: true` for the 5 confirmed URLs, `false` for
  `euraxess-algeria` (kept `enabled: false`), `notes` on every row;
- **feed table**: 3 sample entries (or live entries) with parsed `published` dates;
- **Tavily table**: 3 hits — live results, or the labeled sample when offline;
- **usage**: `made` grows only on real calls, repeats show `cached`, the cap demo shows
  `made: 1` + `skipped: 1` (a query cached by an earlier run shows `cached` instead —
  cache hits are free and bypass the cap);
- nothing raises — dead feed, missing key or failing API all become messages.